# 04  Comparabilidad temporal y monetaria


- **Ventana principal: enero–agosto del tercer año** (2022 vs 2026): mismo punto del mandato, mismos meses, ambos años con Ley de Garantías presidencial y 2022 completo en SECOP II.
- Los 17 meses (abr-2021 a ago-2022) pasan a **sensibilidad**: abril de 2021 es el mes en que la Alcaldía empezó a usar SECOP II y la cobertura de 2021 no está validada.
- Cada diferencia recibe un dictamen de **robustez**: solo es publicable si conserva el signo en todas las ventanas.
- Honorarios: **serie semestral en pesos constantes**, no solo un contraste entre alcaldes.

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().resolve()
# Busca la raíz del proyecto (carpeta que contiene funciones/secop_utils.py)
for _p in [RAIZ, *RAIZ.parents][:6]:
    if (_p / "funciones" / "secop_utils.py").exists():
        RAIZ = _p
        break
else:
    raise FileNotFoundError("No se encontró funciones/secop_utils.py; abra el notebook dentro del proyecto")
sys.path.insert(0, str(RAIZ / "funciones"))
import secop_utils as su

VERSION_NB = "04.v2.0"
ETAPA = "04_comparabilidad"
SALIDA = su.carpeta_etapa(RAIZ, ETAPA)
UMBRAL_ROBUSTEZ_PCT = 3.0   # diferencias menores se consideran estables
man01, r01 = su.abrir_etapa(RAIZ, "01_base")
man03, r03 = su.abrir_etapa(RAIZ, "03_cps")
estado_cob = su.leer_json(r01["estado_cobertura"])
b = su.leer_csv(r03["base_cps"])
print("Cobertura 2021 validada:", estado_cob["cobertura_2021_validada"], "| primer mes central con >=50 contratos:", estado_cob["primer_mes_central_50_o_mas"])

Cobertura 2021 validada: False | primer mes central con >=50 contratos: 2021-04-01


## IPC oficial DANE (archivo fijado por hash)

In [3]:
RUTA_IPC = RAIZ / "datos" / "referencias" / "ipc_dane" / "anex-IPC-Indices-ago2026.xlsx"
SHA_IPC = "bd622e4c35b80c83085cbd24b6fdc282e578c0c2b15ac6f9eae332ddff52e264"
if su.sha256_archivo(RUTA_IPC) != SHA_IPC:
    raise RuntimeError("El anexo IPC no coincide con la versión fijada.")
bruto = pd.read_excel(RUTA_IPC, sheet_name="IndicesIPC", header=None)
fila = bruto.index[bruto.iloc[:, 0].astype(str).str.strip().str.lower().eq("mes")][0]
tabla = bruto.iloc[fila + 1: fila + 13].copy()
tabla.columns = bruto.iloc[fila].tolist()
MESES = {m: i for i, m in enumerate(["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto",
                                    "Septiembre", "Octubre", "Noviembre", "Diciembre"], 1)}
ipc = tabla.melt(id_vars=tabla.columns[0], var_name="anio", value_name="ipc_indice")
ipc["anio"] = pd.to_numeric(ipc["anio"], errors="coerce")
ipc["num_mes"] = ipc.iloc[:, 0].map(MESES)
ipc["ipc_indice"] = pd.to_numeric(ipc["ipc_indice"], errors="coerce")
ipc = ipc.dropna(subset=["anio", "num_mes", "ipc_indice"]).query("2020 <= anio <= 2026")
ipc["mes"] = pd.to_datetime(dict(year=ipc["anio"].astype(int), month=ipc["num_mes"].astype(int), day=1))
ipc = ipc[["mes", "ipc_indice"]].sort_values("mes").reset_index(drop=True)
for mes, esperado in {"2020-01-01": 104.24, "2022-01-01": 113.26, "2026-08-01": 160.42}.items():
    assert np.isclose(ipc.loc[ipc["mes"].eq(mes), "ipc_indice"].item(), esperado), mes
MES_BASE = ipc["mes"].max()
IPC_BASE = ipc["ipc_indice"].iloc[-1]
print(f"IPC {ipc['mes'].min():%Y-%m} a {MES_BASE:%Y-%m}; pesos constantes de {MES_BASE:%Y-%m}")

IPC 2020-01 a 2026-08; pesos constantes de 2026-08


## Universo comparable: CPS estrictos de la Alcaldía central con identidad apta

In [4]:
c = b.loc[b["es_central"] & b["apto_persona"]].copy()
c["mes_firma"] = c["fecha_firma"].dt.to_period("M").dt.to_timestamp()
c = c.merge(ipc.rename(columns={"mes": "mes_firma"}), on="mes_firma", how="left", validate="many_to_one")
c["valor_mensual_equiv_real"] = c["valor_mensual_equiv"] * IPC_BASE / c["ipc_indice"]
sin_ipc = int((c["valor_mensual_equiv"].notna() & c["ipc_indice"].isna()).sum())
print(f"Contratos: {len(c):,} | con valor mensual sin IPC (meses posteriores a {MES_BASE:%Y-%m}): {sin_ipc}")

Contratos: 26,128 | con valor mensual sin IPC (meses posteriores a 2026-08): 27


## Ventanas

In [5]:
estado_17 = ("SENSIBILIDAD" if estado_cob["cobertura_2021_validada"] else "SENSIBILIDAD_COBERTURA_2021_NO_VALIDADA")
VENTANAS = pd.DataFrame([
    ("anio3_ene_ago", "Alfonso Eljach", "2022-01-01", "2022-09-01", "PRINCIPAL"),
    ("anio3_ene_ago", "Jonathan Vásquez", "2026-01-01", "2026-09-01", "PRINCIPAL"),
    ("meses_16_32", "Alfonso Eljach", "2021-04-01", "2022-09-01", estado_17),
    ("meses_16_32", "Jonathan Vásquez", "2025-04-01", "2026-09-01", estado_17),
    ("meses_17_32", "Alfonso Eljach", "2021-05-01", "2022-09-01", estado_17),
    ("meses_17_32", "Jonathan Vásquez", "2025-05-01", "2026-09-01", estado_17),
], columns=["ventana", "administracion", "inicio", "fin_excl", "rol"])
VENTANAS[["inicio", "fin_excl"]] = VENTANAS[["inicio", "fin_excl"]].apply(pd.to_datetime)
VENTANAS["dias"] = (VENTANAS["fin_excl"] - VENTANAS["inicio"]).dt.days
assert VENTANAS.groupby("ventana")["dias"].nunique().eq(1).all(), "Exposición desigual"
partes = []
for v in VENTANAS.itertuples(index=False):
    m = c["fecha_firma"].ge(v.inicio) & c["fecha_firma"].lt(v.fin_excl)
    partes.append(c.loc[m].assign(ventana=v.ventana, administracion=v.administracion, rol=v.rol))
cv = pd.concat(partes, ignore_index=True)
assert not cv.duplicated(["ventana", "administracion", "id_contrato"]).any()
VENTANAS

,ventana,administracion,inicio,fin_excl,rol,dias
0,anio3_ene_ago,Alfonso Eljach,2022-01-01,2022-09-01,PRINCIPAL,243
1,anio3_ene_ago,Jonathan Vásquez,2026-01-01,2026-09-01,PRINCIPAL,243
2,meses_16_32,Alfonso Eljach,2021-04-01,2022-09-01,SENSIBILIDAD_COBERTURA_2021_NO_VALIDADA,518
3,meses_16_32,Jonathan Vásquez,2025-04-01,2026-09-01,SENSIBILIDAD_COBERTURA_2021_NO_VALIDADA,518
4,meses_17_32,Alfonso Eljach,2021-05-01,2022-09-01,SENSIBILIDAD_COBERTURA_2021_NO_VALIDADA,488
5,meses_17_32,Jonathan Vásquez,2025-05-01,2026-09-01,SENSIBILIDAD_COBERTURA_2021_NO_VALIDADA,488


## Indicadores por ventana (volumen, duración, cobertura, valor)

In [6]:
def indicadores(g, v):
    cob = su.cobertura_por_persona(g.loc[g["apto_intervalo"]], v.inicio, v.fin_excl)
    fila = {
        "contratos": len(g),
        "personas": g["documento_identidad"].nunique(),
        "contratos_por_persona": len(g) / max(g["documento_identidad"].nunique(), 1),
        "duracion_mediana_dias": su.mediana(g.loc[g["apto_intervalo"], "duracion_dias_incl"]),
        "cobertura_mediana_meses": su.mediana(cob["meses_cubiertos"]),
        "pct_personas_mitad_ventana_o_mas": 100 * (cob["dias_cubiertos"] >= v.dias / 2).mean() if len(cob) else np.nan,
        "pct_ambiguo": 100 * g["subtipo_cps"].eq("Ambiguo").mean(),
    }
    for sub, etiqueta in [("Profesional", "prof"), ("Apoyo a la gestión", "apoyo")]:
        s = g.loc[g["apto_valor_mensual"] & g["subtipo_cps"].eq(sub)]
        fila[f"mediana_real_{etiqueta}"] = su.mediana(s["valor_mensual_equiv_real"])
        fila[f"mediana_real_{etiqueta}_ge90"] = su.mediana(s.loc[s["duracion_dias_incl"].ge(90), "valor_mensual_equiv_real"])
        fila[f"mediana_nominal_{etiqueta}"] = su.mediana(s["valor_mensual_equiv"])
        fila[f"n_{etiqueta}"] = len(s)
    return fila

filas = []
for v in VENTANAS.itertuples(index=False):
    g = cv.loc[cv["ventana"].eq(v.ventana) & cv["administracion"].eq(v.administracion)]
    filas.append({"ventana": v.ventana, "administracion": v.administracion, "rol": v.rol, **indicadores(g, v)})
resumen = pd.DataFrame(filas)
resumen.set_index(["ventana", "administracion"]).T.round(1)

ventana                            anio3_ene_ago                   \
administracion                    Alfonso Eljach Jonathan Vásquez   
rol                                    PRINCIPAL        PRINCIPAL   
contratos                                   3378             4133   
personas                                    2819             3123   
contratos_por_persona                   1.198297         1.323407   
duracion_mediana_dias                      126.0            120.0   
cobertura_mediana_meses                 4.073922         3.942505   
pct_personas_mitad_ventana_o_mas       50.894775        38.084489   
pct_ambiguo                             0.976909         0.290346   
mediana_real_prof                 4993371.171357   4225597.672053   
mediana_real_prof_ge90            5001861.396991   4225597.672053   
mediana_nominal_prof              3527524.834437   4058333.333333   
n_prof                                      1602             1842   
mediana_real_apoyo                2766418.369197   2640998.545033   
mediana_real_apoyo_ge90           2809166.555917   2640998.545033   
mediana_nominal_apoyo             1995901.639344   2536458.333333   
n_apoyo                                     1701             2215   

ventana                                                       meses_16_32  \
administracion                                             Alfonso Eljach   
rol                               SENSIBILIDAD_COBERTURA_2021_NO_VALIDADA   
contratos                                                            6216   
personas                                                             3381   
contratos_por_persona                                            1.838509   
duracion_mediana_dias                                               120.0   
cobertura_mediana_meses                                          5.650924   
pct_personas_mitad_ventana_o_mas                                26.439869   
pct_ambiguo                                                      0.965251   
mediana_real_prof                                          4996340.578672   
mediana_real_prof_ge90                                     5001861.396991   
mediana_nominal_prof                                        3501027.03103   
n_prof                                                               2934   
mediana_real_apoyo                                         2874085.437637   
mediana_real_apoyo_ge90                                    2855051.759241   
mediana_nominal_apoyo                                      1995901.639344   
n_apoyo                                                              3070   

ventana                                                                    \
administracion                                           Jonathan Vásquez   
rol                               SENSIBILIDAD_COBERTURA_2021_NO_VALIDADA   
contratos                                                            7796   
personas                                                             4277   
contratos_por_persona                                            1.822773   
duracion_mediana_dias                                               106.0   
cobertura_mediana_meses                                          4.533881   
pct_personas_mitad_ventana_o_mas                                24.171952   
pct_ambiguo                                                      0.256542   
mediana_real_prof                                           4241109.22556   
mediana_real_prof_ge90                                     4250617.714738   
mediana_nominal_prof                                       4031456.953642   
n_prof                                                               3406   
mediana_real_apoyo                                         2656636.071711   
mediana_real_apoyo_ge90                                    2656636.071711   
mediana_nominal_apoyo                                      2536458.333333   
n_apoyo                                                      

## Dictamen de robustez
`ROBUSTO`: mismo signo, |cambio| ≥ 3 % y magnitud estable (máx/mín ≤ 2) en las tres ventanas. `DIRECCION_ROBUSTA_MAGNITUD_SENSIBLE`: mismo signo pero la cifra depende de la ventana. `ESTABLE`: |cambio| < 3 % en todas. `NO_ROBUSTO`: cambia de signo o de magnitud según la ventana.

In [7]:
METRICAS = ["contratos", "personas", "contratos_por_persona", "duracion_mediana_dias", "cobertura_mediana_meses",
            "pct_personas_mitad_ventana_o_mas", "mediana_real_prof", "mediana_real_apoyo",
            "mediana_real_prof_ge90", "mediana_real_apoyo_ge90"]
largo = resumen.melt(id_vars=["ventana", "administracion", "rol"], value_vars=METRICAS, var_name="metrica", value_name="valor")
ancho = largo.pivot_table(index=["metrica", "ventana"], columns="administracion", values="valor").reset_index()
ancho["cambio_pct"] = [su.cambio_pct(a, j) for a, j in zip(ancho["Alfonso Eljach"], ancho["Jonathan Vásquez"])]
ancho.loc[ancho["metrica"].str.startswith("pct_"), "cambio_pct"] = (
    ancho["Jonathan Vásquez"] - ancho["Alfonso Eljach"])  # en puntos porcentuales


def dictamen(s):
    s = s.dropna()
    if len(s) < 3:
        return "INSUFICIENTE"
    if (s.abs() < UMBRAL_ROBUSTEZ_PCT).all():
        return "ESTABLE"
    if (s >= UMBRAL_ROBUSTEZ_PCT).all() or (s <= -UMBRAL_ROBUSTEZ_PCT).all():
        # misma dirección; si la magnitud más que se duplica entre ventanas, la cifra no es estable
        return "ROBUSTO" if s.abs().max() / s.abs().min() <= 2 else "DIRECCION_ROBUSTA_MAGNITUD_SENSIBLE"
    return "NO_ROBUSTO"

robustez = (ancho.pivot(index="metrica", columns="ventana", values="cambio_pct")
            .assign(dictamen=lambda t: t.apply(dictamen, axis=1)).reset_index())
robustez["validacion_subtipo"] = np.where(robustez["metrica"].str.contains("prof|apoyo"),
                                          man03["conteos"]["validacion_subtipo"], "No aplica")
robustez.round(1)

ventana,metrica,anio3_ene_ago,meses_16_32,meses_17_32,dictamen,validacion_subtipo
0,cobertura_mediana_meses,-3.2,-19.8,-23.7,DIRECCION_ROBUSTA_MAGNITUD_SENSIBLE,No aplica
1,contratos,22.4,25.4,23.5,ROBUSTO,No aplica
2,contratos_por_persona,10.4,-0.9,-1.9,NO_ROBUSTO,No aplica
3,duracion_mediana_dias,-4.8,-11.7,-15.8,DIRECCION_ROBUSTA_MAGNITUD_SENSIBLE,No aplica
4,mediana_real_apoyo,-4.5,-7.6,-7.1,ROBUSTO,SIN_VALIDACION_MANUAL
5,mediana_real_apoyo_ge90,-6.0,-6.9,-6.8,ROBUSTO,SIN_VALIDACION_MANUAL
6,mediana_real_prof,-15.4,-15.1,-15.2,ROBUSTO,SIN_VALIDACION_MANUAL
7,mediana_real_prof_ge90,-15.5,-15.0,-15.0,ROBUSTO,SIN_VALIDACION_MANUAL
8,pct_personas_mitad_ventana_o_mas,-12.8,-2.3,-5.0,NO_ROBUSTO,No aplica
9,personas,10.8,26.5,25.9,DIRECCION_ROBUSTA_MAGNITUD_SENSIBLE,No aplica


## Honorarios: serie semestral en pesos constantes
Permite ver si un cambio ocurre en el paso de gobierno o es una tendencia previa.

In [8]:
s = c.loc[c["apto_valor_mensual"] & c["ipc_indice"].notna()].copy()
s["semestre"] = s["fecha_firma"].dt.year.astype(str) + "-S" + np.where(s["fecha_firma"].dt.month <= 6, "1", "2")
serie_honorarios = (s.groupby(["semestre", "subtipo_cps"])
                    .agg(contratos=("id_contrato", "size"), mediana_real=("valor_mensual_equiv_real", "median"),
                         p25_real=("valor_mensual_equiv_real", lambda x: x.quantile(.25)),
                         p75_real=("valor_mensual_equiv_real", lambda x: x.quantile(.75)),
                         mediana_nominal=("valor_mensual_equiv", "median"))
                    .reset_index())
serie_honorarios["administracion"] = np.where(serie_honorarios["semestre"] < "2024", "Alfonso Eljach", "Jonathan Vásquez")
serie_honorarios = serie_honorarios.loc[serie_honorarios["contratos"].ge(100)]  # semestres con base suficiente
serie_honorarios.pivot(index="semestre", columns="subtipo_cps", values="mediana_real").round(-3)

subtipo_cps,Apoyo a la gestión,Profesional
semestre,,
2021-S1,2927000.0,4857000.0
2021-S2,2926000.0,5120000.0
2022-S1,2587000.0,4949000.0
2022-S2,2615000.0,4612000.0
2023-S1,2393000.0,4252000.0
2023-S2,2612000.0,4155000.0
2024-S1,2778000.0,4558000.0
2024-S2,2794000.0,4471000.0
2025-S1,2767000.0,4452000.0


In [9]:
# ¿Cuándo ocurre la caída? Tramo dentro de Alfonso vs salto en el cambio de gobierno.
piv = serie_honorarios.pivot(index="semestre", columns="subtipo_cps", values="mediana_real")
tramos = []
for sub in ["Profesional", "Apoyo a la gestión"]:
    x = piv[sub].dropna()
    tramos += [
        {"subtipo": sub, "tramo": "Primer semestre observado → 2023-S2 (dentro de Alfonso)",
         "cambio_pct": su.cambio_pct(x.iloc[0], x.get("2023-S2"))},
        {"subtipo": sub, "tramo": "2023-S2 → 2024-S1 (cambio de gobierno)",
         "cambio_pct": su.cambio_pct(x.get("2023-S2"), x.get("2024-S1"))},
        {"subtipo": sub, "tramo": "2024-S1 → último semestre (dentro de Jonathan)",
         "cambio_pct": su.cambio_pct(x.get("2024-S1"), x.iloc[-1])},
    ]
tramos_honorarios = pd.DataFrame(tramos)
tramos_honorarios.round(1)

,subtipo,tramo,cambio_pct
0,Profesional,Primer semestre observado → 2023-S2 (dentro de...,-14.4
1,Profesional,2023-S2 → 2024-S1 (cambio de gobierno),9.7
2,Profesional,2024-S1 → último semestre (dentro de Jonathan),-3.1
3,Apoyo a la gestión,Primer semestre observado → 2023-S2 (dentro de...,-10.8
4,Apoyo a la gestión,2023-S2 → 2024-S1 (cambio de gobierno),6.3
5,Apoyo a la gestión,2024-S1 → último semestre (dentro de Jonathan),1.9


## Misma persona, mismo subtipo (ventana principal) contexto, no causal

In [10]:
vp = cv.loc[cv["ventana"].eq("anio3_ene_ago") & cv["apto_valor_mensual"] & cv["valor_mensual_equiv_real"].notna()]
ind = (vp.groupby(["documento_identidad", "subtipo_cps", "administracion"])["valor_mensual_equiv_real"]
       .median().unstack("administracion").dropna())
dentro = []
for sub, g in ind.groupby(level="subtipo_cps"):
    d = 100 * (g["Jonathan Vásquez"] / g["Alfonso Eljach"] - 1)
    dentro.append({"subtipo": sub, "personas_pareadas": len(d), "mediana_cambio_pct": d.median(),
                   "pct_personas_con_aumento": 100 * (d > 0).mean(),
                   "lectura": "Misma persona con 4 años de diferencia: incluye inflación, experiencia y cambio de funciones."})
dentro_persona = pd.DataFrame(dentro)
dentro_persona.round(1)

,subtipo,personas_pareadas,mediana_cambio_pct,pct_personas_con_aumento,lectura
0,Apoyo a la gestión,217,1.2,52.5,Misma persona con 4 años de diferencia: incluy...
1,Profesional,265,-12.6,20.8,Misma persona con 4 años de diferencia: incluy...


## Controles y cierre

In [11]:
ctl = su.Controles()
ctl.agregar("Exposición igual por ventana", bool(VENTANAS.groupby("ventana")["dias"].nunique().eq(1).all()), True)
ctl.agregar("Solo Alcaldía central", int((~cv["es_central"]).sum()), 0)
ctl.agregar("Sin duplicados ventana-contrato", int(cv.duplicated(["ventana", "administracion", "id_contrato"]).sum()), 0)
ctl.agregar("Ventana principal con IPC completo",
            int(cv.loc[cv["ventana"].eq("anio3_ene_ago") & cv["apto_valor_mensual"], "ipc_indice"].isna().sum()), 0)
ctl.agregar("Cobertura 2021 validada (sensibilidad 17 meses)", estado_cob["cobertura_2021_validada"], True, "Importante")
ctl.agregar("Subtipo validado manualmente", man03["conteos"]["validacion_subtipo"], "VALIDADA", "Importante")
tabla_ctl = ctl.tabla()
display(tabla_ctl)

ids_ventana = cv[["ventana", "administracion", "rol", "id_contrato"]]
salidas = {
    "ventanas": su.guardar_csv(VENTANAS, SALIDA / "registro_ventanas.csv"),
    "contratos_ventana": su.guardar_csv(ids_ventana, SALIDA / "contratos_por_ventana.csv"),
    "resumen_ventanas": su.guardar_csv(resumen, SALIDA / "resumen_ventanas.csv"),
    "robustez": su.guardar_csv(robustez, SALIDA / "robustez_comparaciones.csv"),
    "serie_honorarios": su.guardar_csv(serie_honorarios, SALIDA / "serie_honorarios_semestral.csv"),
    "tramos_honorarios": su.guardar_csv(tramos_honorarios, SALIDA / "tramos_honorarios.csv"),
    "dentro_persona": su.guardar_csv(dentro_persona, SALIDA / "dentro_persona_principal.csv"),
    "ipc": su.guardar_csv(ipc, SALIDA / "ipc_dane_mensual.csv"),
    "controles": su.guardar_csv(tabla_ctl, SALIDA / "controles_04.csv"),
}
estado = "BLOQUEADO" if ctl.bloqueos() else ("VALIDADO_CON_ALERTAS" if ctl.alertas() else "VALIDADO")
man = su.cerrar_etapa(RAIZ, ETAPA, VERSION_NB, {"01": su.huella_entrada(man01), "03": su.huella_entrada(man03)}, salidas,
                      reglas={"ventana_principal": "Enero-agosto del tercer año: 2022 vs 2026.",
                              "sensibilidad": f"Meses 16-32 y 17-32 ({estado_17}).",
                              "robustez": f"Publicable como diferencia solo si ROBUSTO (umbral {UMBRAL_ROBUSTEZ_PCT} %).",
                              "ipc": f"Pesos constantes de {MES_BASE:%Y-%m}; cruce exacto por mes de firma.",
                              "cobertura": "Días únicos cubiertos por contratos firmados dentro de la ventana."},
                      conteos={"filas_ventana_contrato": len(cv),
                               "dictamenes": robustez.set_index("metrica")["dictamen"].to_dict(),
                               "mes_base_ipc": f"{MES_BASE:%Y-%m}"},
                      estado=estado, alertas=ctl.bloqueos() + ctl.alertas())
if ctl.bloqueos():
    raise RuntimeError(f"Etapa 04 bloqueada: {ctl.bloqueos()}")
print(man["estado"])
robustez[["metrica", "dictamen"]]

,prueba,resultado,esperado,severidad,pasa
0,Exposición igual por ventana,True,True,Crítica,True
1,Solo Alcaldía central,0,0,Crítica,True
2,Sin duplicados ventana-contrato,0,0,Crítica,True
3,Ventana principal con IPC completo,0,0,Crítica,True
4,Cobertura 2021 validada (sensibilidad 17 meses),False,True,Importante,False
5,Subtipo validado manualmente,SIN_VALIDACION_MANUAL,VALIDADA,Importante,False


VALIDADO_CON_ALERTAS


ventana,metrica,dictamen
0,cobertura_mediana_meses,DIRECCION_ROBUSTA_MAGNITUD_SENSIBLE
1,contratos,ROBUSTO
2,contratos_por_persona,NO_ROBUSTO
3,duracion_mediana_dias,DIRECCION_ROBUSTA_MAGNITUD_SENSIBLE
4,mediana_real_apoyo,ROBUSTO
5,mediana_real_apoyo_ge90,ROBUSTO
6,mediana_real_prof,ROBUSTO
7,mediana_real_prof_ge90,ROBUSTO
8,pct_personas_mitad_ventana_o_mas,NO_ROBUSTO
9,personas,DIRECCION_ROBUSTA_MAGNITUD_SENSIBLE
